In [13]:
BRANCH = "main"

In [3]:
# !pip install dagshub dvc

In [14]:
import os
# import dagshub
import traceback
# from google.colab import userdata
import pandas as pd
import numpy as np
import re

In [5]:
# # Step 1: Install and authenticate
# dagshub.auth.add_app_token(userdata.get('DAGSHUB_TOKEN'))

# # Step 2: Mount the storage bucket (Linux/Colab supports this)
# mount_path = str(dagshub.storage.mount("dilminidasunika/movie_recommendation_system"))

# print("Mounted at:", mount_path)
# # Now you can read files directly from mount_path

In [15]:
df = pd.read_csv("../data/TMDB_all_movies.csv")

df

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path,keywords,certification_us
0,2,Ariel,7.100,382.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9999.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,"prison|underdog|helsinki, finland|factory work...",NaN
1,3,Shadows in Paradise,7.300,448.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8840.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,"helsinki, finland|salesclerk|garbage",NR
2,5,Four Rooms,5.905,2851.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,"Quentin Tarantino, Robert Rodriguez, Alexandre...","Phil Parmet, Guillermo Navarro, Rodrigo García...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,6.7,117462.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg,hotel|new year's eve|witch|bet|sperm|hotel roo...,R
3,6,Judgment Night,6.500,385.0,Released,1993-10-15,12136938.0,110.0,21000000.0,tt0107286,...,Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,6.6,21307.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg,"drug dealer|chicago, illinois|escape|one night...",R
4,8,Life in Loops (A Megacities RMX),7.300,34.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,Timo Novotny,Wolfgang Thaler,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg,megacities,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237107,1757483,FIGHT,0.000,0.0,Released,NaN,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237108,1757484,Ahma's Favourite Tree,0.000,0.0,Released,NaN,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237109,1757485,Between Silence and Truth,0.000,0.0,Released,NaN,0.0,0.0,0.0,tt34725314,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237110,1757486,Mimmo the Milker,0.000,0.0,Released,NaN,0.0,8.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,1237102,1237103,1237104,1237105,1237106,1237107,1237108,1237109,1237110,1237111
id,2,3,5,6,8,9,11,12,13,14,...,1757475,1757476,1757477,1757481,1757482,1757483,1757484,1757485,1757486,1757487
title,Ariel,Shadows in Paradise,Four Rooms,Judgment Night,Life in Loops (A Megacities RMX),Sunday in August,Star Wars,Finding Nemo,Forrest Gump,American Beauty,...,Miss Marple : Endless Night,Tempestad en la Cordillera,Finger Guns,Lab Pe Aati Hai Dua by Syeda Areeba Fatima,El Guardián,FIGHT,Ahma's Favourite Tree,Between Silence and Truth,Mimmo the Milker,Usunure Panty
vote_average,7.1,7.3,5.905,6.5,7.3,6.8,8.207,7.82,8.463,7.998,...,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
vote_count,382.0,448.0,2851.0,385.0,34.0,28.0,22739.0,20878.0,30311.0,13270.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
status,Released,Released,Released,Released,Released,Released,Released,Released,Released,Released,...,Released,Released,Released,Released,Released,Released,Released,Released,Released,Released
release_date,1988-10-21,1986-10-17,1995-12-09,1993-10-15,2006-01-01,2004-09-02,1977-05-25,2003-05-30,1994-06-23,1999-09-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
revenue,0.0,0.0,4257354.0,12136938.0,0.0,0.0,775398007.0,940335536.0,677387716.0,356296601.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
runtime,73.0,74.0,98.0,110.0,80.0,15.0,121.0,100.0,142.0,122.0,...,0.0,0.0,10.0,4.0,0.0,0.0,0.0,0.0,8.0,60.0
budget,0.0,0.0,4000000.0,21000000.0,42000.0,0.0,11000000.0,94000000.0,55000000.0,15000000.0,...,0.0,2500.0,0.0,0.0,200.0,0.0,0.0,0.0,0.0,0.0
imdb_id,tt0094675,tt0092149,tt0113101,tt0107286,tt0825671,tt0425473,tt0076759,tt0266543,tt0109830,tt0169547,...,tt2166494,tt33070889,tt44792194,tt44878814,NaN,NaN,NaN,tt34725314,NaN,tt4450898


# Select features

Plot
- overview

Tags
- title
- original_title
- original_language
- genres
- production_companies
- production_countries
- spoken_languages
- cast
- director
- writers
- producers
- music_composer
- keywords



# Preprocessing

In [17]:
selected_df = df[["overview", "title", "original_title", "original_language", "genres", "production_companies", "production_countries", "spoken_languages", "cast", "director", "writers", "producers", "music_composer", "keywords"]]
selected_df

,overview,title,original_title,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords
0,A Finnish man goes to the city to find a job a...,Ariel,Ariel,fi,"Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,NaN,"prison|underdog|helsinki, finland|factory work..."
1,"Nikander, a rubbish collector and would-be ent...",Shadows in Paradise,Varjoja paratiisissa,fi,"Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,Aki Kaurismäki,Mika Kaurismäki,NaN,"helsinki, finland|salesclerk|garbage"
2,It's Ted the Bellhop's first night on the job....,Four Rooms,Four Rooms,en,Comedy,"Miramax, A Band Apart",United States of America,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,hotel|new year's eve|witch|bet|sperm|hotel roo...
3,"Four young friends, while taking a shortcut en...",Judgment Night,Judgment Night,en,"Action, Crime, Drama, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,...",Stephen Hopkins,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,"drug dealer|chicago, illinois|escape|one night..."
4,Timo Novotny labels his new project an experim...,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX),en,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,megacities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237107,NaN,FIGHT,FIGHT,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237108,NaN,Ahma's Favourite Tree,Ahma's Favourite Tree,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237109,NaN,Between Silence and Truth,Between Silence and Truth,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1237110,"Alien cows, hungry for cheddar, invade Planet ...",Mimmo the Milker,Mimmo the Milker,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
selected_df.shape

(1237112, 14)

In [20]:
print("overview empty count:" , int(selected_df["overview"].isna().sum()))
print("title empty count:" , int(selected_df["title"].isna().sum()))
print("original_title empty count:" , int(selected_df["original_title"].isna().sum()))
print("original_language empty count:" , int(selected_df["original_language"].isna().sum()))
print("genres empty count:" , int(selected_df["genres"].isna().sum()))
print("production_companies empty count:" , int(selected_df["production_companies"].isna().sum()))
print("production_countries empty count:" , int(selected_df["production_countries"].isna().sum()))
print("spoken_languages empty count:" , int(selected_df["spoken_languages"].isna().sum()))
print("cast empty count:" , int(selected_df["cast"].isna().sum()))
print("director empty count:" , int(selected_df["director"].isna().sum()))
print("writers empty count:" , int(selected_df["writers"].isna().sum()))
print("producers empty count:" , int(selected_df["producers"].isna().sum()))
print("music_composer empty count:" , int(selected_df["music_composer"].isna().sum()))
print("keywords empty count:" , int(selected_df["keywords"].isna().sum()))

overview empty count: 190094
title empty count: 15
original_title empty count: 14
original_language empty count: 0
genres empty count: 332563
production_companies empty count: 638685
production_countries empty count: 459616
spoken_languages empty count: 451661
cast empty count: 390939
director empty count: 198290
writers empty count: 609316
producers empty count: 810295
music_composer empty count: 1094316
keywords empty count: 886181


In [21]:
selected_df[(selected_df['title'].isna() & ~selected_df['original_title'].isna())]

,overview,title,original_title,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords
1091344,"Through the eyes of Era, a resilient yet fragi...",NaN,Jerëm,sq,Drama,"Thumbs Up, Tunnelfilm","Kosovo, Macedonia, Albania",shqip,"Suela Bako, Genc Jakupi, Shpëtim Kastrati, Ire...",Ismet Sijarina,Ismet Sijarina,NaN,NaN,NaN


## Remove empty Data rows

In [22]:
cleaned_df = selected_df.dropna(subset=['title'])
selected_df.shape

(1237112, 14)

In [23]:
cleaned_df = cleaned_df.dropna(subset=['genres','production_companies','production_countries','cast','director','writers','producers','music_composer','keywords'], how='all')
cleaned_df.shape

(1168877, 14)

In [24]:
cleaned_df

,overview,title,original_title,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords
0,A Finnish man goes to the city to find a job a...,Ariel,Ariel,fi,"Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,NaN,"prison|underdog|helsinki, finland|factory work..."
1,"Nikander, a rubbish collector and would-be ent...",Shadows in Paradise,Varjoja paratiisissa,fi,"Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,Aki Kaurismäki,Mika Kaurismäki,NaN,"helsinki, finland|salesclerk|garbage"
2,It's Ted the Bellhop's first night on the job....,Four Rooms,Four Rooms,en,Comedy,"Miramax, A Band Apart",United States of America,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,hotel|new year's eve|witch|bet|sperm|hotel roo...
3,"Four young friends, while taking a shortcut en...",Judgment Night,Judgment Night,en,"Action, Crime, Drama, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,...",Stephen Hopkins,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,"drug dealer|chicago, illinois|escape|one night..."
4,Timo Novotny labels his new project an experim...,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX),en,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,megacities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237061,NaN,Turma 101,Turma 101,pt,NaN,NaN,NaN,NaN,NaN,Wagner Novais,Wagner Novais,NaN,NaN,NaN
1237076,From the master filmmaker behind Daydream and ...,Midnight Sun Dream Episode 2,白夜夢 2話,en,NaN,NaN,NaN,NaN,NaN,Tetsuji Takechi,NaN,NaN,NaN,NaN
1237084,NaN,Siento para ver,Siento para ver,es,NaN,NaN,NaN,NaN,NaN,Nino Yeverino Heyser,NaN,NaN,NaN,NaN
1237086,NaN,SOCAÚVO; EL SAUCE FÚNEBRE,SOCAÚVO; EL SAUCE FÚNEBRE,es,NaN,NaN,NaN,NaN,Perla Villarello,NaN,NaN,NaN,NaN,NaN


In [25]:
print("overview empty count:" , int(cleaned_df["overview"].isna().sum()))
print("title empty count:" , int(cleaned_df["title"].isna().sum()))
print("original_title empty count:" , int(cleaned_df["original_title"].isna().sum()))
print("original_language empty count:" , int(cleaned_df["original_language"].isna().sum()))
print("genres empty count:" , int(cleaned_df["genres"].isna().sum()))
print("production_companies empty count:" , int(cleaned_df["production_companies"].isna().sum()))
print("production_countries empty count:" , int(cleaned_df["production_countries"].isna().sum()))
print("spoken_languages empty count:" , int(cleaned_df["spoken_languages"].isna().sum()))
print("cast empty count:" , int(cleaned_df["cast"].isna().sum()))
print("director empty count:" , int(cleaned_df["director"].isna().sum()))
print("writers empty count:" , int(cleaned_df["writers"].isna().sum()))
print("producers empty count:" , int(cleaned_df["producers"].isna().sum()))
print("music_composer empty count:" , int(cleaned_df["music_composer"].isna().sum()))
print("keywords empty count:" , int(cleaned_df["keywords"].isna().sum()))

overview empty count: 170274
title empty count: 0
original_title empty count: 0
original_language empty count: 0
genres empty count: 264337
production_companies empty count: 570454
production_countries empty count: 391387
spoken_languages empty count: 389054
cast empty count: 322710
director empty count: 130063
writers empty count: 541086
producers empty count: 742064
music_composer empty count: 1026082
keywords empty count: 817951


## Check Unique Tags

In [19]:
genres = (
    cleaned_df['genres']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique genres:", genres.nunique())
print(genres.value_counts())

Number of unique genres: 19
genres
Drama              303902
Documentary        235046
Comedy             180196
Horror              74704
Animation           74558
Music               65787
Romance             65557
Thriller            63338
Action              54618
Crime               43109
Family              35950
TV Movie            33590
Fantasy             31266
Adventure           29224
Science Fiction     27908
Mystery             27854
History             22996
War                 13375
Western             10298
Name: count, dtype: int64


In [20]:
company = (
    cleaned_df['production_companies']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique companies:", company.nunique())
print(company.value_counts())

Number of unique companies: 218322
production_companies
BBC                          4159
ARTE                         3734
ONF | NFB                    3313
Warner Bros. Pictures        3163
Columbia Pictures            3001
                             ... 
Bodo Production                 1
Tarangini Film Syndicate        1
One Spotlight Productions       1
24P                             1
Igel Media AG                   1
Name: count, Length: 218322, dtype: int64


In [21]:
countries = (
    cleaned_df['production_countries']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique countries:", countries.nunique())
print(countries.value_counts())

Number of unique countries: 250
production_countries
United States of America    193052
France                       58021
United Kingdom               55175
Japan                        51029
Germany                      48464
                             ...  
Anguilla                         2
Cocos  Islands                   2
East Timor                       2
Burma                            2
Zaire                            1
Name: count, Length: 250, dtype: int64


In [22]:
original_language = (
    cleaned_df['original_language']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique original_language:", original_language.nunique())
print(original_language.value_counts())

Number of unique original_language: 177
original_language
en    552718
fr     76696
es     74301
ja     56817
de     53649
       ...  
io         1
kj         1
nr         1
ii         1
ng         1
Name: count, Length: 177, dtype: int64


In [23]:
spoken_languages = (
    cleaned_df['spoken_languages']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique spoken_languages:", spoken_languages.nunique())
print(spoken_languages.value_counts())

Number of unique spoken_languages: 92
spoken_languages
English          297146
Español           64062
Français          62987
日本語               54329
No Language       48217
                  ...  
Qafar af              5
Kirundi               5
Aragonés              4
Bislama               2
Upastawakaēna         2
Name: count, Length: 92, dtype: int64


In [24]:
def split_cast(s):
    parts = re.split(r',(?!\s*(?:Jr|jr|Sr|sr|II|III|IV|V)\.?\s*(?:,|$))\s*', s)
    return [p.strip() for p in parts if p.strip()]

In [25]:
cast = (
    cleaned_df['cast']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique cast:", cast.nunique())
print(cast.value_counts())

Number of unique cast: 2209246
cast
Jr.                  1220
Mel Blanc            1125
Bess Flowers          751
Eric Roberts          702
Jagathy Sreekumar     687
                     ... 
Devon Culture           1
Antwione Saunders       1
Mikko Remes             1
Jyrki Olsonen           1
Eino Kuusela            1
Name: count, Length: 2209246, dtype: int64


In [26]:
cast2 = cleaned_df['cast'].dropna().apply(split_cast).explode().str.strip()

print("Number of unique cast:", cast2.nunique())
print(cast2.value_counts())
suspects = cast2.value_counts()[cast2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', ''])]
print(suspects)

Number of unique cast: 2209377
cast
Mel Blanc                     1125
Bess Flowers                   751
Eric Roberts                   702
Jagathy Sreekumar              687
Brahmanandam                   680
                              ... 
Francisco González Martín        1
Francisca González Jiménez       1
Kenneth Phillips                 1
Kim Minji                        1
Afonso Morales                   1
Name: count, Length: 2209377, dtype: int64
cast
Jr.    1
Name: count, dtype: int64


In [27]:
director = (
    cleaned_df['director']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique director:", director.nunique())
print(director.value_counts())
suspects_dir = director.value_counts()[director.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_dir)

Number of unique director: 440422
director
Dave Fleischer                639
Gérard Courant                502
D.W. Griffith                 467
Kevin Dunn                    464
Paul Terry                    413
                             ... 
Itzel García de Jesús           1
Alejandro Cárdenas              1
Juan Carlos Lelo de Larrea      1
Teenaa Kaur                     1
Lisa Brougham                   1
Name: count, Length: 440422, dtype: int64
director
Jr.    174
        19
Sr.      4
Jr       2
III      2
V        1
Name: count, dtype: int64


In [28]:
director2 = (
    cleaned_df['director']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique director:", director2.nunique())
print(director2.value_counts())
suspects_dir2 = director2.value_counts()[director2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_dir2)

Number of unique director: 440422
director
Dave Fleischer                639
Gérard Courant                502
D.W. Griffith                 467
Kevin Dunn                    464
Paul Terry                    413
                             ... 
Juan Carlos Lelo de Larrea      1
Teenaa Kaur                     1
Jeffrey D. King                 1
John Spellos                    1
Sergey Pankov                   1
Name: count, Length: 440422, dtype: int64
director
V    1
Name: count, dtype: int64


In [29]:
writers = (
    cleaned_df['writers']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique writers:", writers.nunique())
print(writers.value_counts())
suspects_wri = writers.value_counts()[writers.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V','X', ''])]
print(suspects_wri)

Number of unique writers: 417543
writers
William Shakespeare    776
Stephen King           532
Anton Chekhov          351
Vince McMahon          305
Edgar Allan Poe        266
                      ... 
Tekin Akmansoy           1
Vince Mascoli            1
Martin Itković           1
王盈希                      1
Jazz Pitcairn            1
Name: count, Length: 417543, dtype: int64
writers
Jr.    206
         9
V        4
Sr.      3
Jr       3
II       1
III      1
Name: count, dtype: int64


In [30]:
writers2 = (
    cleaned_df['writers']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique writers:", writers2.nunique())
print(writers2.value_counts())
suspects_wri2 = writers2.value_counts()[writers2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_wri2)

Number of unique writers: 417544
writers
William Shakespeare           776
Stephen King                  532
Anton Chekhov                 351
Vince McMahon                 305
Edgar Allan Poe               266
                             ... 
Aquiles Saraiva                 1
Michael Baumbruck               1
Andrea Howard                   1
Skye Wingo                      1
Miccaella Lazo Del Rosario      1
Name: count, Length: 417544, dtype: int64
writers
V    3
Name: count, dtype: int64


In [31]:
producers = (
    cleaned_df['producers']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique producers:", producers.nunique())
print(producers.value_counts())
suspects_prod = producers.value_counts()[producers.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_prod)

Number of unique producers: 361633
producers
Louis Lumière       964
Auguste Lumière     886
Walt Disney         628
Tom Berry           614
Sebastian Battro    536
                   ... 
PieEnPared            1
Austine Abella        1
Daisuke Uriu          1
Troy Schmidt          1
Patrick Meenan        1
Name: count, Length: 361633, dtype: int64
producers
Jr.    212
        10
Sr.      6
III      4
V        2
Jr       1
Name: count, dtype: int64


In [32]:
producers2 = (
    cleaned_df['producers']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique producers:", producers2.nunique())
print(producers2.value_counts())
suspects_prod2 = producers2.value_counts()[producers2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_prod2)

Number of unique producers: 361642
producers
Louis Lumière          964
Auguste Lumière        886
Walt Disney            628
Tom Berry              614
Sebastian Battro       536
                      ... 
Kelly Garrett-Orona      1
Papatinho                1
Chris Plater             1
Boris Tosic              1
Jonathan Rej             1
Name: count, Length: 361642, dtype: int64
Series([], Name: count, dtype: int64)


In [33]:
music_composer = (
    cleaned_df['music_composer']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique music_composer:", music_composer.nunique())
print(music_composer.value_counts())
suspects_music = music_composer.value_counts()[music_composer.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_music)

Number of unique music_composer: 47738
music_composer
Ilaiyaraaja        726
Ennio Morricone    327
Masaru Satō        263
Zdeněk Liška       255
Chūji Kinoshita    249
                  ... 
Marcos Arévalo       1
Tural Bakuvi         1
Sigrid Kemp          1
Murat Engin          1
Mattia D’Ambros      1
Name: count, Length: 47738, dtype: int64
music_composer
Jr.    5
Name: count, dtype: int64


In [34]:
music_composer2 = (
    cleaned_df['music_composer']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique music_composer:", music_composer2.nunique())
print(music_composer2.value_counts())
suspects_music2 = music_composer2.value_counts()[music_composer2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', ''])]
print(suspects_music2)

Number of unique music_composer: 47737
music_composer
Ilaiyaraaja        726
Ennio Morricone    327
Masaru Satō        263
Zdeněk Liška       255
Chūji Kinoshita    249
                  ... 
Marcos Arévalo       1
Tural Bakuvi         1
Sigrid Kemp          1
Murat Engin          1
Mattia D’Ambros      1
Name: count, Length: 47737, dtype: int64
Series([], Name: count, dtype: int64)


In [35]:
keywords = (
    cleaned_df['keywords']
    .dropna()
    .str.split('|')
    .explode()
    .str.strip()
)

print("Number of unique keywords:", keywords.nunique())
print(keywords.value_counts())
suspects_key = keywords.value_counts()[keywords.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_key)

Number of unique keywords: 77768
keywords
short film                38117
woman director            16409
based on novel or book     8374
lgbt                       7241
concert                    6188
                          ...  
migratory                     1
manuel antin                  1
images d'archives             1
consommation                  1
criminalité                   1
Name: count, Length: 77768, dtype: int64
Series([], Name: count, dtype: int64)


In [36]:
def has_lone_jr(s):
    parts = split_cast(s)
    return any(p.strip() in ('Jr.', 'Jr', 'jr.', 'jr') for p in parts)

offenders = cleaned_df['cast'].dropna()
offenders = offenders[offenders.apply(has_lone_jr)]
print(offenders.head(20).tolist())

['Jr.']


In [37]:
mask = cleaned_df['cast'].str.contains(r'\bjr\b', case=False, na=False, regex=True)
print(mask.sum())   # how many rows actually matched — do this FIRST, always

17215


In [38]:
with pd.option_context('display.max_colwidth', None):
    display(cleaned_df.loc[mask, 'cast'].sample(20))

,cast
343170,"Renato Del Prado, Raymart Santiago, Ace Cruz, Karen Espino, Nonoy De Guzman, Vincent Berba, Roel De Villa, Lauro Delgado Jr., Joseph Abul, Dan Fernandez, Rando Almanzor, Dexter Doria, Toni Daya, Manjo del Mundo, Jack Barri"
89864,"Hallie Foote, William Converse-Roberts, Michael Higgins, Rochelle Oliver, Richard Jenkins, Jeanne McCarthy, Steven Hill, Irma P. Hall, Matthew Broderick, Carol Goodheart, Horton Foote Jr."
48784,"Eddie Bo Smith Jr., Karen Williams, Jean-Louis Roux, Stacy Keach, David La Haye, Cynda Williams, Raven Dauda, Gil Bellows, Graeme Somerville, Jane Wheeler, Sylvia Stewart, Omari Newton, Chris Williams, Vanessa Williams, Jillian Hervey, Lise Roy, Diahann Carroll, Kevin Jubinville, Susannah Hoffmann, Lisa Bronwyn Moore"
206193,"Katherine Helen Fisher, Antoine Silverman, Lindsay Kesselman, Caitlin Scranton, Melanie Russell, Tomás Cruz, Sharon Milanese, Sarah Hillmon, Patrick John O'Neill, Shakirah Stewart, Lonnie Poupard Jr, Jasper Newell, Philip Anderson, Hai-Ting Chinn, Gregory R. Purnhagen, Anne Lewis, Charles Williams, Jason Charles Walker, Helga Davis, Matt Pardo, John Sorensen-Jolink, Kate Moran, Michèle A. Eaton, John Kawa, Katie Dorn, Joe Damon Chappel, Kate Maroney, Solange Merdinian"
104053,"Chick Chandler, Otto Kruger, Gladden James, Paul Harvey, Don 'Red' Barry, Addison Richards, Tom Powers, Jay Novello, Joseph Crehan, Lynne Roberts, Henry H. Daniels Jr., Kenne Duncan"
485825,"Anne Gentry, Toni L. Mitchell, Paul Bartels, Jim Milstead, Selena Villalobos, Maya Rasheed-Moore, Carl Cabrese, Alex Kass, Frank Mejia, Audrey Beth, Eric Roberts, Lance Sandoval, Greg Garcia, Corey Gifft Jr., Alvaro Fallas, Jonathan Garcia, Garry Cox, Willie M. Gray, Liam Hawley, Clinton Towne, Salahaddin Keskin, Roberto Cordero Jr., Paris Rasheed-Moore, Alexandria Stevens, Anderson Johnson III, Ben Stone, Jeff Beaupre, Phillip Penza, Ramsey Penza, Kevin Mallaire, Mathias Schlogl, Big Daddy Kane, Michael Montgomery, Lorenzo Lamas, Jeff Holder"
377967,"Jaret Reddick, Lanie Taylor, Travis Edwards Jr., Sheril Rodgers, Mike Norris, Grey Acuna, Nathan Amir, April Hartman, Rashaun Sibley"
792614,"Will Ashcroft, Channy Mitchell, Yasmin Duursma, Alwyn Davey Jr, Jayden Davey"
32984,"Richard Eden, Joe Dain, Brian Peck, Eric Roberts, Theresa Russell, Gavin Harrison, Tom Ward, James Marsden, Dan Cortese, Grant Cramer, Ed Dollison, Leah Best, Brendan Williams, Alyssa Milano, Frank Stallone Jr., Rex Linn, Gary Sievers, Terry D. Seago, Chip Heller, Michael Stadvec"
67527,"David Cramer, Gary McCleery, Robert G. Denison, Pat Studstill, Rhetta Greene, Larry Swartz, John Chappell, Scott Paulin, Anne Ramsey, Sondra Blake, Barry Corbin, Robert S. Telford, Med Flory, Dennis Redfield, Chris Mulkey, Hal Holbrook, Tom Rayhall, John Dennis Johnston, Buck Young, Elizabeth Macey, Cal Haynes, Susan James Berger, James Whitmore Jr., Nancy Malone, Robert Phalen, Billy C. Chandler, John H. Fields, Anthony Edwards, Dixie Carter, Jim Antonio, Richard Beauchamp, Sean Penn, Jennifer Jason Leigh, James Brewer"


## Test

In [40]:
test_df = pd.DataFrame({
    'cast': [
        'Robert Downey, Jr., Tom Hanks, Samuel L. Jackson III',
        'Meryl Streep, Denzel Washington',
        np.nan,
        '',
        'Martin Luther King, Sr.',
        'Will Smith'
    ],
    'genres': [
        'Action, Adventure',
        'Drama',
        np.nan,
        '',
        'Comedy, Romance',
        np.nan
    ]
})

test_df

,cast,genres
0,"Robert Downey, Jr., Tom Hanks, Samuel L. Jacks...","Action, Adventure"
1,"Meryl Streep, Denzel Washington",Drama
2,NaN,NaN
3,,
4,"Martin Luther King, Sr.","Comedy, Romance"
5,Will Smith,NaN


In [41]:
# --- BEFORE: check dtype ---
print(test_df['cast'].dtype)           # object
print(type(test_df['cast'].iloc[0]))   # <class 'str'>
print(type(test_df['cast'].iloc[2]))   # <class 'float'> (NaN)

# --- CONVERT using split_cast ---
cast = test_df['cast'].fillna('').apply(split_cast)

# --- AFTER: check dtype ---
print(cast.dtype)             # still object
print(type(cast.iloc[0]))     # <class 'list'>
print(type(cast.iloc[2]))     # <class 'list'> -- was NaN, now []

# --- INSPECT ROW BY ROW ---
for i in range(len(test_df)):
    print(i, '|', repr(test_df['cast'].iloc[i]), '->', cast.iloc[i])

# --- SANITY CHECK ---
assert cast.apply(lambda x: isinstance(x, list)).all()
print("all rows are lists ✓")

object
<class 'str'>
<class 'float'>
object
<class 'list'>
<class 'list'>
0 | 'Robert Downey, Jr., Tom Hanks, Samuel L. Jackson III' -> ['Robert Downey, Jr.', 'Tom Hanks', 'Samuel L. Jackson III']
1 | 'Meryl Streep, Denzel Washington' -> ['Meryl Streep', 'Denzel Washington']
2 | nan -> []
3 | '' -> []
4 | 'Martin Luther King, Sr.' -> ['Martin Luther King, Sr.']
5 | 'Will Smith' -> ['Will Smith']
all rows are lists ✓


In [42]:
cast

,cast
0,"[Robert Downey, Jr., Tom Hanks, Samuel L. Jack..."
1,"[Meryl Streep, Denzel Washington]"
2,[]
3,[]
4,"[Martin Luther King, Sr.]"
5,[Will Smith]


## DB Upload with `to_sql`

In [26]:
def split_cast(s):
    parts = re.split(r',(?!\s*(?:Jr|jr|Sr|sr|II|III|IV|V)\.?\s*(?:,|$))\s*', s)
    return [p.strip() for p in parts if p.strip()]

In [32]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="nd2002",
    password="As123@we",
    host="localhost",
    port=15432,
    database="moviedb"
)

engine = create_engine(url)

DDL = """
CREATE TABLE IF NOT EXISTS movies (
    id SERIAL PRIMARY KEY,
    title TEXT NOT NULL,
    original_title TEXT,
    original_language TEXT,
    genres TEXT[],
    production_companies TEXT[],
    production_countries TEXT[],
    spoken_languages TEXT[],
    cast_members TEXT[],
    director TEXT[],
    writers TEXT[],
    producers TEXT[],
    music_composer TEXT[],
    keywords TEXT[],
    overview TEXT
);

CREATE INDEX IF NOT EXISTS idx_movies_genres ON movies USING GIN (genres);
CREATE INDEX IF NOT EXISTS idx_movies_cast ON movies USING GIN (cast_members);
CREATE INDEX IF NOT EXISTS idx_movies_keywords ON movies USING GIN (keywords);
CREATE INDEX IF NOT EXISTS idx_movies_title ON movies (title);
"""

with engine.begin() as conn:
    conn.execute(text(DDL))

In [33]:
cleaned_df['genres'] = cleaned_df['genres'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['production_companies'] = cleaned_df['production_companies'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['production_countries'] = cleaned_df['production_countries'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['spoken_languages'] = cleaned_df['spoken_languages'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['cast_members'] = cleaned_df['cast'].fillna('').apply(split_cast)
cleaned_df['director'] = cleaned_df['director'].fillna('').apply(split_cast)
cleaned_df['writers'] = cleaned_df['writers'].fillna('').apply(split_cast)
cleaned_df['producers'] = cleaned_df['producers'].fillna('').apply(split_cast)
cleaned_df['music_composer'] = cleaned_df['music_composer'].fillna('').apply(split_cast)
cleaned_df['keywords'] = cleaned_df['keywords'].fillna('').apply(
    lambda s: [g.strip() for g in s.split('|') if g.strip()]
)
cleaned_df = cleaned_df.drop(columns=['cast'])

In [34]:
cleaned_df.to_sql(
    'movies',
    engine,
    if_exists='append',
    index=False,
    method='multi',
    chunksize=5000
)

1168877